# 04 — Three-Class Shop Classification

Self-contained notebook: no `%run` dependency on project scripts.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import yaml

# In Colab, change only this path after mounting Google Drive.
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "src" else Path(".").resolve()
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
with CONFIG_PATH.open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

Labels must be manually reviewed. Labels generated mechanically from Score_ThamKhao would cause circular evaluation because the classifier uses overlapping features.

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

source = OUTPUT / "labeling" / "Data_Labeled.csv"
if not source.exists():
    raise FileNotFoundError("Human-labeled Data_Labeled.csv is required")
data = pd.read_csv(source)
features = ["Years_Active", "Total_Products", "Follower_Count_k", "Rating_Average", "Chat_Response_Rate_pct", "Authentic_Review_Rate"]
data = data.dropna(subset=["Target_Label", *features]).copy()
data["Target_Label"] = data["Target_Label"].astype(int)
display(data["Target_Label"].value_counts().sort_index().rename("Shop_Count"))
if len(data) < 10 or data["Target_Label"].nunique() < 2 or data["Target_Label"].value_counts().min() < 2:
    raise ValueError("Insufficient labeled shops/classes for an honest stratified evaluation")

In [ ]:
train, test = train_test_split(data, test_size=CONFIG["classification"]["test_size"], random_state=CONFIG["project"]["random_state"], stratify=data["Target_Label"])
model = RandomForestClassifier(n_estimators=CONFIG["classification"]["n_estimators"], random_state=CONFIG["project"]["random_state"], class_weight="balanced")
model.fit(train[features], train["Target_Label"])
prediction = model.predict(test[features])
probability = model.predict_proba(test[features]).max(axis=1)
accuracy = accuracy_score(test["Target_Label"], prediction)
macro = precision_recall_fscore_support(test["Target_Label"], prediction, average="macro", zero_division=0)
weighted_f1 = precision_recall_fscore_support(test["Target_Label"], prediction, average="weighted", zero_division=0)[2]
metrics = {
    "accuracy": accuracy, "macro_precision": macro[0], "macro_recall": macro[1], "macro_f1": macro[2],
    "weighted_f1": weighted_f1, "target_accuracy": CONFIG["classification"]["accuracy_target"],
    "pass_accuracy_requirement": bool(accuracy > CONFIG["classification"]["accuracy_target"]),
}
display(pd.DataFrame([metrics]))
print(classification_report(test["Target_Label"], prediction, zero_division=0))
display(pd.DataFrame(confusion_matrix(test["Target_Label"], prediction)))

In [ ]:
result = pd.DataFrame({"Shop_ID": test["Shop_ID"], "Actual_Label": test["Target_Label"], "Predicted_Label": prediction, "Predicted_Probability": probability})
result.to_csv(OUTPUT / "reports" / "Classification_Result.csv", index=False)
with (OUTPUT / "reports" / "classification_metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)
joblib.dump(model, OUTPUT / "models" / "random_forest_shop_reputation.joblib")
display(result)